In [5]:
import pandas as pd
from pyrosm import OSM
import folium
from folium.plugins import MarkerCluster
import gc
import os

In [ ]:
# 1. กำหนดค่าพิกัด Bounding Box ตีกรอบพื้นที่เฉพาะกรุงเทพฯ
osm_filepath = "../../data/raw/thailand-260703.osm.pbf"
bkk_bbox = [100.30, 13.45, 100.95, 13.95]

print("1. 🚀 กำลังเชื่อมต่อไฟล์ดิบออฟไลน์และจำกัดพื้นที่ประมวลผล...")
osm = OSM(osm_filepath, bounding_box=bkk_bbox)

print("2. 🏗️ กำลังสกัดข้อมูลคู่แข่ง (กรองเฉพาะหมวดหมู่ธุรกิจอาหารตั้งแต่ต้นน้ำเพื่อประหยัดแรม)...")
# 💡 เปลี่ยนเป็น custom_filter ตาม Syntax ที่ถูกต้องของ pyrosm เพื่อสกัดเฉพาะกลุ่มร้านค้าคู่แข่ง (Supply Side) ทันที
target_competitors = ['restaurant', 'cafe', 'fast_food', 'bar']
pois_gdf = osm.get_pois(custom_filter={"amenity": target_competitors})

# ดักจับเงื่อนไขความปลอดภัยป้องกันโค้ดระเบิด
if pois_gdf is None or len(pois_gdf) == 0:
    print("❌ ข้อผิดพลาด: ไม่พบข้อมูลจุดสนใจในขอบเขตพื้นที่ที่กำหนด")
else:
    print(f"✅ สกัดออบเจกต์คู่แข่งออกมาได้สำเร็จ: {len(pois_gdf)} แถว")

    print("3. 📌 กำลังคำนวณสกัดพิกัดตัวเลขละติจูด/ลองจิจูดออกจากออบเจกต์เชิงพื้นที่...")
    pois_gdf['latitude'] = pois_gdf.geometry.centroid.y
    pois_gdf['longitude'] = pois_gdf.geometry.centroid.x

    print("4. 📊 แปลงโครงสร้างข้อมูลเป็นตาราง Pandas DataFrame และหั่นส่วนที่กินแรมทิ้ง...")
    df_pois_flat = pd.DataFrame(pois_gdf)
    
    # 💡 เคลียร์แรมระหว่างทาง: ทำลายคอลัมน์ geometry (ออบเจกต์รูปทรงภูมิศาสตร์) ทันทีเพราะกินเนื้อที่แรมมหาศาล
    if 'geometry' in df_pois_flat.columns:
        df_pois_flat = df_pois_flat.drop(columns=['geometry'])

    # ระบบป้องกันคอลัมน์หล่น (ถ้าในไฟล์ไม่มีข้อมูลฟีเจอร์นั้น ให้สร้างค่าเริ่มต้นดักไว้ก่อนเพื่อไม่ให้เกิด KeyError)
    expected_competitor_cols = {
        'name': 'Unknown Business',
        'amenity': None,
        'cuisine': 'general',            # สไตล์ประเภทอาหาร (เช่น coffee_shop, thai, japanese)
        'opening_hours': 'not_specified', # เวลาเปิด-ปิด
        'addr:street': 'not_specified',
        'addr:postcode': 'not_specified'
    }
    for col, default_val in expected_competitor_cols.items():
        if col not in df_pois_flat.columns:
            df_pois_flat[col] = default_val

    print("5. 🧹 กำลังทำความสะอาดข้อมูล คัดกรองซ้ำ และล้างจุดพิกัดทับซ้อน (Deduplication)...")
    # ทำการกรองย้ำความชัวร์ในโครงสร้าง DataFrame
    df_comp_filtered = df_pois_flat[df_pois_flat['amenity'].isin(target_competitors)].copy()
    df_comp_filtered['name'] = df_comp_filtered['name'].fillna(df_comp_filtered['amenity'])

    # สกัดเอาเฉพาะคอลัมน์แกนหลักไปทำโมเดล Scoring และพ่นหน้า Dashboard
    df_competitors_final = df_comp_filtered[[
        'name', 
        'amenity', 
        'cuisine', 
        'opening_hours', 
        'addr:street', 
        'addr:postcode', 
        'latitude', 
        'longitude'
    ]].copy()

    # ปรับชื่อคอลัมน์ให้อยู่ในฟอร์แมต Schema มาตรฐานของระบบซอฟต์แวร์
    df_competitors_final.columns = [
        'name', 
        'amenity_type', 
        'cuisine', 
        'opening_hours', 
        'street', 
        'postcode', 
        'latitude', 
        'longitude'
    ]

    # ล้างจุดพิกัดที่เป็นค่าว่าง (NaN) และพิกัดร้านค้าที่ปักหมุดซ้ำกันเป๊ะๆ ออกจากสารบบ
    df_competitors_final = df_competitors_final.dropna(subset=['latitude', 'longitude'])
    df_competitors_final = df_competitors_final.drop_duplicates(subset=['latitude', 'longitude'])

    print("\n--- 📊 สรุปจำนวนยอดข้อมูลคู่แข่ง (Supply Assets) แยกตามหมวดหมู่ธุรกิจ ---")
    print(df_competitors_final['amenity_type'].value_counts())
    print(f"คงเหลือข้อมูลร้านคู่แข่งเคลียร์สะอาดพร้อมวิเคราะห์เชิงลึก: {len(df_competitors_final)} ร้าน")

    # 6. บันทึกเดตาสะอาดก้อนที่สองดิ่งตรงลงสู่โฟลเดอร์ตามโครงสร้างที่จัดระเบียบไว้
    os.makedirs("../../data/interim", exist_ok=True)
    output_competitor_file = "../../data/interim/bangkok_competitors_clean.json"
    df_competitors_final.to_json(output_competitor_file, orient='records', force_ascii=False, indent=4)
    print(f"💾 🎉 บันทึกสำเร็จ! จัดเก็บไฟล์โครงสร้างวิเคราะห์ไว้ที่: {output_competitor_file}")

1. 🚀 กำลังเชื่อมต่อไฟล์ดิบออฟไลน์และจำกัดพื้นที่ประมวลผล...
2. 🏗️ กำลังสกัดข้อมูลคู่แข่ง (กรองเฉพาะหมวดหมู่ธุรกิจอาหารตั้งแต่ต้นน้ำเพื่อประหยัดแรม)...
✅ สกัดออบเจกต์คู่แข่งออกมาได้สำเร็จ: 9148 แถว
3. 📌 กำลังคำนวณสกัดพิกัดตัวเลขละติจูด/ลองจิจูดออกจากออบเจกต์เชิงพื้นที่...
4. 📊 แปลงโครงสร้างข้อมูลเป็นตาราง Pandas DataFrame และหั่นส่วนที่กินแรมทิ้ง...
5. 🧹 กำลังทำความสะอาดข้อมูล คัดกรองซ้ำ และล้างจุดพิกัดทับซ้อน (Deduplication)...

--- 📊 สรุปจำนวนยอดข้อมูลคู่แข่ง (Supply Assets) แยกตามหมวดหมู่ธุรกิจ ---
amenity_type
restaurant    5282
cafe          2449
fast_food      763
bar            652
Name: count, dtype: int64
คงเหลือข้อมูลร้านคู่แข่งเคลียร์สะอาดพร้อมวิเคราะห์เชิงลึก: 9146 ร้าน
💾 🎉 บันทึกสำเร็จ! จัดเก็บไฟล์โครงสร้างวิเคราะห์ไว้ที่: ../cleaned-assets/bangkok_competitors_clean.json


/tmp/ipykernel_22310/1980449215.py:20: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois_gdf['latitude'] = pois_gdf.geometry.centroid.y
/tmp/ipykernel_22310/1980449215.py:21: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois_gdf['longitude'] = pois_gdf.geometry.centroid.x


In [7]:
# 7. 🧹 กระบวนการกวาดล้างขยะในหน่วยความจำแบบเด็ดขาด (Garbage Collection)
print("\n🧹 กำลังสั่งการทำลายตัวแปรยักษ์ทั้งหมดออกจากพื้นที่หน่วยความจำ...")
if 'pois_gdf' in locals(): del pois_gdf
if 'df_pois_flat' in locals(): del df_pois_flat
if 'df_comp_filtered' in locals(): del df_comp_filtered
if 'df_competitors_final' in locals(): del df_competitors_final
gc.collect()

print("🤖 [สถานะ: ปลอดภัย] ล้างแรมเกลี้ยงหมดจด 100% เครื่องคอมพิวเตอร์ของคุณเบาหวิวเรียบร้อยครับ!")


🧹 กำลังสั่งการทำลายตัวแปรยักษ์ทั้งหมดออกจากพื้นที่หน่วยความจำ...
🤖 [สถานะ: ปลอดภัย] ล้างแรมเกลี้ยงหมดจด 100% เครื่องคอมพิวเตอร์ของคุณเบาหวิวเรียบร้อยครับ!


In [ ]:
# 8. 🗺️ โหลดข้อมูลเฉพาะที่บันทึกเสร็จแล้วมา 2,000 จุด เพื่อวาดแผนที่แสดงผลสวยๆ โดยไม่จองแรมค้าง
print("\n🗺️ กำลังประมวลผลวาดแผนที่คู่แข่ง... (ดึง 2,000 ร้านแรกมาพล็อตเพื่อป้องกันระบบค้าง)")
m_comp = folium.Map(location=[13.7563, 100.5018], zoom_start=11)
marker_cluster_comp = MarkerCluster().add_to(m_comp)

comp_color_map = {
    'restaurant': 'orange',
    'cafe': 'purple',
    'fast_food': 'darkred',
    'bar': 'black'
}

# เปิดอ่านไฟล์ที่บันทึกไว้ตะกี้เฉพาะหัวตารางมาพล็อต
df_map_view = pd.read_json("../data/interim/bangkok_competitors.json")

for idx, row in df_map_view.head(2000).iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"<b>{row['name']}</b><br>ประเภท: {row['amenity_type']}<br>สไตล์อาหาร: {row['cuisine']}",
        icon=folium.Icon(color=comp_color_map.get(row['amenity_type'], 'gray'), icon='shopping-cart')
    ).add_to(marker_cluster_comp)

# ทำลายตัวแปรตารางวาดแผนที่เพื่อคืนแรมรอบสอง
del df_map_view
gc.collect()

print("🎉 ทุกกระบวนการเสร็จสิ้น 100% แผนที่คู่แข่งแสดงผลด้านล่างครับ!")
m_comp



🗺️ กำลังประมวลผลวาดแผนที่คู่แข่ง... (ดึง 2,000 ร้านแรกมาพล็อตเพื่อป้องกันระบบค้าง)
🎉 ทุกกระบวนการเสร็จสิ้น 100% แผนที่คู่แข่งแสดงผลด้านล่างครับ!
